In [ ]:
import seaborn as sns
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mticker

In [ ]:
paths = ["BPR", "VAE", "NCF", "SVD"]

In [ ]:
common_metrics = [
    'KLD_test', 'NDCG_test', 'recall_test', 'Total MRR',
    'normal_PL', 'novelty_normal',
    'Coverage', 'normalized_DPF',
    'normalized_exposure_1', 'normalized_exposure_2'
]

model_configs = {
    'BPR': {
        'csv_files': [
            'BPR_1M_LDP.csv', 'BPR_1M_DPSGD.csv',  # csv files that include our results
            'BPR_Yelp_LDP.csv', 'BPR_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["1M_BPR_50", "BPR_YELP"],
        'normal_names': ['1M_BPR_50', "", 'BPR_YELP'],
        'model_labels': ['BPR_1M', "", 'BPR_YELP'],
        'metrics': common_metrics
    },
    'SVD': {
        'csv_files': [
            'SVD_1M_LDP.csv', 'SVD_1M_DPSGD.csv',
            'SVD_Yelp_LDP.csv', 'SVD_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["1M_SVD_50", "SVD_Yelp_50"],
        'normal_names': ['1M_SVD_50', "", 'SVD_Yelp_50'],
        'model_labels': ['SVD_1M', "", 'SVD_Yelp'],
        'metrics': common_metrics
    },
    'VAE': {
        'csv_files': [
            'VAE_1M_LDP.csv', 'VAE_1M_DPSGD.csv',
            'VAE_Yelp_LDP.csv', 'VAE_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["vae_1M", "vae_Yelp"],
        'normal_names': ['vae_1M', "", 'vae_Yelp'],
        'model_labels': ['VAE_1M', "", 'VAE_Yelp'],
        'metrics': [m for m in common_metrics if m != 'Total MRR']
    },
    'NCF': {
        'csv_files': [
            'Deep_1M_LDP.csv', 'Deep_1M_DPSGD.csv',
            'Deep_Yelp_LDP.csv', 'Deep_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["Deep_1M_50", 'Yelp_Deep_50'],
        'normal_names': ['Deep_1M_50', "", 'Yelp_Deep_50'],
        'model_labels': ['NCF_1M', "", 'NCF_Yelp'],
        'metrics': common_metrics
    }
}

In [ ]:
# with different style for dashed lines and save one by one

sns.set_style("whitegrid")


def plot_user_based_second(df_dpsgd, df_dpsgd_yelp, normal_models, metric, title, scales, path_save_base, dpsgd=False):
    state = 'DPSGD' if dpsgd else 'DP'
    df_normal_1M = normal_models[0]
    df_normal_Yelp = normal_models[2]

    df_normal_1M.loc[:, 'epsilon'] = 0
    df_normal_Yelp.loc[:, 'epsilon'] = 0

    normal_name_1 = df_normal_1M['model_name'].unique().tolist()[0]
    normal_name_2 = df_normal_Yelp['model_name'].unique().tolist()[0]

    label_name = metric.split('_')[0].upper()

    df_dpsgd = df_dpsgd.loc[df_dpsgd['model_name'] != normal_name_1]
    df_dpsgd_yelp = df_dpsgd_yelp.loc[df_dpsgd_yelp['model_name'] != normal_name_2]

    datasets = ['1M', 'Yelp']

    def prepare_combined_data(data, dataset_name):
        prepared_data = []
        for user_type in ['niche_user', 'blockbuster', 'diverse']:
            key = f'{user_type} {metric} Normal'
            if key not in data.columns:
                key = f'{user_type} {metric}'
            if key in data.columns:
                temp_df = data[['epsilon', key]].copy()
                temp_df = temp_df.rename(columns={key: 'value'})
                temp_df['user_type'] = user_type
                temp_df['dataset'] = dataset_name
                prepared_data.append(temp_df)
        return pd.concat(prepared_data)

    data1_results = prepare_combined_data(df_dpsgd, datasets[0])
    data2_results = prepare_combined_data(df_dpsgd_yelp, datasets[1])

    combined_data = pd.concat([data1_results, data2_results], ignore_index=True)
    combined_data['user_type_dataset'] = combined_data['user_type'] + ' ' + combined_data['dataset']

    # Define color, marker, and linestyle mappings
    paired_palette = sns.color_palette("Paired")
    user_type_palette = {
        'niche_user 1M': paired_palette[0],  # Dark Blue
        'blockbuster 1M': paired_palette[2],  # Dark Red
        'diverse 1M': paired_palette[4],  # Dark Green
        'niche_user Yelp': paired_palette[1],  # Light Blue
        'blockbuster Yelp': paired_palette[3],  # Light Red
        'diverse Yelp': paired_palette[5]  # Light Green
    }

    user_type_markers = {
        'niche_user 1M': 'o',  # Circle
        'blockbuster 1M': 's',  # Square
        'diverse 1M': 'D',  # Diamond
        'niche_user Yelp': 'o',  # Circle
        'blockbuster Yelp': 's',  # Square
        'diverse Yelp': 'D'  # Diamond
    }

    # Different dashed styles for non-private baselines
    user_type_linestyles = {
        'niche_user 1M': '--',  # Short dashes
        'blockbuster 1M': '-.',  # Dash-dot
        'diverse 1M': ':',  # Dotted
        'niche_user Yelp': '--',  # Short dashes
        'blockbuster Yelp': '-.',  # Dash-dot
        'diverse Yelp': ':'  # Dotted
    }
    legend_name_mapping = {
        'niche_user 1M': 'niche (1M)',
        'blockbuster 1M': 'blockbuster (1M)',
        'diverse 1M': 'diverse (1M)',
        'niche_user Yelp': 'niche (Yelp)',
        'blockbuster Yelp': 'blockbuster (Yelp)',
        'diverse Yelp': 'diverse (Yelp)',
        'niche_user 1M (non-private)': 'niche (1M) Non-Private',
        'blockbuster 1M (non-private)': 'blockbuster (1M) Non-Private',
        'diverse 1M (non-private)': 'diverse (1M) Non-Private',
        'niche_user Yelp (non-private)': 'niche (Yelp) Non-Private',
        'blockbuster Yelp (non-private)': 'blockbuster (Yelp) Non-Private',
        'diverse Yelp (non-private)': 'diverse (Yelp) Non-Private', }
    # ---- Create separate figures ----
    fig, ax = plt.subplots(figsize=(3, 2.5))  # Adjusted to match the original half-size
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.set_xlim(0.008, 406)  # min/max epsilon over all datasets as of 2025-07-17
    for user_type in ['niche_user', 'blockbuster', 'diverse']:
        for dataset in datasets:
            user_key = f"{user_type} {dataset}"
            subset_data = combined_data[combined_data['user_type_dataset'] == user_key]

            if not subset_data.empty:
                label = legend_name_mapping.get(user_key, user_key)
                sns.lineplot(
                    data=subset_data,
                    x='epsilon', y='value', ax=ax,
                    errorbar=('ci', 95), err_style='bars', err_kws={'fmt': '.', 'ecolor': 'black'},
                    marker=user_type_markers[user_key],  # Assign custom marker
                    markersize=6,
                    linestyle='solid',
                    color=user_type_palette[user_key],
                    label=label  # Legend entry
                )

    # Plot non-private baselines with different dashed styles
    for user_type in ['niche_user', 'blockbuster', 'diverse']:
        key = f'{user_type} {metric} Normal'
        if key not in df_normal_1M.columns:
            key = f'{user_type} {metric}'
        mean_value_1M = df_normal_1M[key].mean()
        mean_value_Yelp = df_normal_Yelp[key].mean()

        label_1M = legend_name_mapping.get(f'{user_type} 1M (non-private)', f'{user_type} 1M (non-private)')
        label_Yelp = legend_name_mapping.get(f'{user_type} Yelp (non-private)', f'{user_type} Yelp (non-private)')

        ax.axhline(mean_value_1M, color=user_type_palette[f'{user_type} 1M'], linewidth=1,
                   linestyle=user_type_linestyles[f'{user_type} 1M'], label=label_1M)
        ax.axhline(mean_value_Yelp, color=user_type_palette[f'{user_type} Yelp'], linewidth=1,
                   linestyle=user_type_linestyles[f'{user_type} Yelp'], label=label_Yelp)

    ax.set_xlabel('Epsilon')
    ax.set_ylabel(f'{plot_names[metric]}')
    #ax.set_title(f'{plot_names[metric]} ({title}) - {state}')
    ax.grid(True)
    ax.legend()
    ax.get_legend().remove()
    ax.set_xscale('log', base=10)
    locmin = mticker.LogLocator(base=10, subs=np.arange(0.1, 1, 0.1), numticks=10)
    ax.xaxis.set_minor_locator(locmin)

    # Adjust Y-limits based on metric
    y_min, _ = ax.get_ylim()
    if metric == 'normalized_DPF':
        ax.set_ylim(-1, 1)
    elif metric == 'Novelty':
        ax.set_ylim(y_min, 30)
    else:
        ax.set_ylim(y_min, scales[metric] + 0.1)

    sns.despine()
    plt.tight_layout()
    save_path = path_save_base.replace(".pdf", f"_{state}.pdf")
    plt.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close(fig)

    print(f"Saved: {save_path}")


In [ ]:

for path in paths:

    dataset = model_configs[path]
    df_dpsgd = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][1]))
    df_dpsgd_yelp = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][3]))
    df_normal_1M = df_dpsgd.loc[df_dpsgd['model_name'] == dataset['normal_model_name'][0]]
    df_normal_yelp = df_dpsgd_yelp.loc[df_dpsgd_yelp['model_name'] == dataset['normal_model_name'][1]]
    # this is the same for all models
    normal_models = [df_normal_1M, "", df_normal_yelp]
    if path == "VAE":
        metrics = ['KLD_test', 'NDCG_test', 'recall_test',
                   'normal_PL', 'novelty_normal',
                   'Coverage', 'normalized_DPF',
                   'normalized_exposure_1', 'normalized_exposure_2']
    metric_user_based = ['Novelty', 'PL', 'KLD', 'NCDG', 'normalized_DPF', 'normalized_coverage_category_I1',
                         'normalized_coverage_category_I2']
    scales = {}

    for metric in metric_user_based:
        max_scale = float('-inf')
        for i in range(len(dataset['csv_files'])):
            data = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][i]))

            for user_type in ['niche_user', 'blockbuster', 'diverse']:
                key = f'{user_type} {metric} Normal'
                if key not in data.columns:
                    key = f'{user_type} {metric}'
                middle_max = max(data[key])
                if middle_max > max_scale:
                    max_scale = middle_max
            scales[metric] = max_scale
    plot_names = {'Novelty': 'Novelty', 'PL': 'PL', 'KLD': 'KLD', 'NCDG': 'NCDG', 'normalized_DPF': 'DPF',
                  'normalized_coverage_category_I1': 'Exposure_category_I1',
                  'normalized_coverage_category_I2': 'Exposure_category_I2'}

    if path == 'BPR':
        name = 'BPR'
    elif path == 'NCF':
        name = 'NCF'
    elif path == 'SVD':
        name = 'SVD'
    elif path == 'VAE':
        name = 'VAE'
    print(path)

    new_path = os.path.join(path, 'user_based')
    if not os.path.exists(new_path):
        os.makedirs(new_path)

    #with PdfPages(path_save) as pdf:
    for i, metric in enumerate(metric_user_based):
        df_mechanism1 = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][0]))
        df_mechanism1 = df_mechanism1[df_mechanism1['epsilon'] > 0]
        df_mechanism2 = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][2]))
        df_mechanism2 = df_mechanism2[df_mechanism2['epsilon'] > 0]
        df_mechanism3 = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][1]))
        df_mechanism3 = df_mechanism3[df_mechanism3['epsilon'] > 0]
        df_mechanism4 = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][3]))
        df_mechanism4 = df_mechanism4[df_mechanism4['epsilon'] > 0]
        file_name = f"{name}_UserBased_{metric}.pdf"
        save_path = os.path.join(new_path, file_name)
        plot_user_based_second(df_mechanism3, df_mechanism4, normal_models, metric, name, scales, save_path, dpsgd=True)
        plot_user_based_second(df_mechanism1, df_mechanism2, normal_models, metric, name, scales, save_path,
                               dpsgd=False)

